# Sand and Dust Storm (SDS) Analysis — Dubai
Methodology based on: *Remote sensing-based sand and dust storm analysis using MODIS data* (2022)
https://www.tandfonline.com/doi/full/10.1080/22797254.2022.2093278

In [49]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='sds-analysis-493219')

print("Connected to Google Earth Engine!")

Connected to Google Earth Engine!


## 1. Area of Interest: Dubai

In [62]:

# ======================================================
# Define ROI: Dubai using geoBoundaries ADM2
# ======================================================
adm2 = ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")

dubaiFC = (
    adm2
    .filter(ee.Filter.eq('shapeGroup', 'ARE'))  # UAE ISO3 = ARE
    .filter(ee.Filter.eq('shapeName', 'Dubai'))
)

roi = dubaiFC.geometry()
print('Dubai features found:', dubaiFC.size().getInfo())


Dubai features found: 1


### Map: Area of Interest

In [63]:

# ======================================================
# Display the Area of Interest
# ======================================================
aoi_map = geemap.Map(basemap="HYBRID")
aoi_map.centerObject(roi, 9)

aoi_map.addLayer(
    dubaiFC,
    {'color': '#FF6600', 'fillColor': '#FF660033'},
    'Dubai AOI'
)

aoi_map


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

## 2. Load Datasets & Define Helpers

In [64]:
# ======================================================
# Load MODIS datasets
# ======================================================
mod09 = ee.ImageCollection("MODIS/061/MOD09GA")
lc    = ee.ImageCollection("MODIS/061/MCD12Q1")
dem   = ee.Image("USGS/SRTMGL1_003")

# ======================================================
# QA masking + reflectance scaling
# ======================================================
def scale_and_mask_MOD09GA(img):
    """Scale SR bands to reflectance and apply cloud mask."""
    b = img.select([
        "sur_refl_b01", "sur_refl_b02", "sur_refl_b03", "sur_refl_b04",
        "sur_refl_b05", "sur_refl_b06", "sur_refl_b07"
    ]).multiply(0.0001)

    # Cloud screening via state_1km bits 0-1: 0=clear, 3=assumed clear
    state = img.select("state_1km")
    cloud_state = state.bitwiseAnd(3)
    clear = cloud_state.eq(0).Or(cloud_state.eq(3))

    return (
        img.addBands(b, None, True)
           .updateMask(clear)
           .copyProperties(img, img.propertyNames())
    )


# ======================================================
# Spectral indices
# ======================================================
def add_indices(img):
    """
    Compute SDS-relevant spectral indices from MOD09GA bands.
    Band mapping (MODIS Terra Surface Reflectance):
      B1=Red(620-670nm)  B2=NIR(841-876nm)  B3=Blue(459-479nm)
      B4=Green(545-565nm) B5=SWIR1(1230nm)  B6=SWIR2(1628nm)  B7=SWIR3(2105nm)
    """
    B1 = img.select("sur_refl_b01")  # Red
    B2 = img.select("sur_refl_b02")  # NIR
    B3 = img.select("sur_refl_b03")  # Blue
    B4 = img.select("sur_refl_b04")  # Green
    B6 = img.select("sur_refl_b06")  # SWIR2
    B7 = img.select("sur_refl_b07")  # SWIR3

    # NDDI: Normalized Difference Dust Index — key SDS indicator
    # High values indicate dust/sand aerosol presence
    nddi = B7.subtract(B3).divide(B7.add(B3)).rename("NDDI")

    # MBSCI: Modified Bare Soil Composition Index — surface sediment source
    mbsci = B4.add(B1).divide(B6.add(B7)).rename("MBSCI")

    # NDVI: Vegetation cover (inverse proxy for dust source availability)
    ndvi = B2.subtract(B1).divide(B2.add(B1)).rename("NDVI")

    # NDWI: Water index (McFeeters)
    ndwi = B4.subtract(B2).divide(B4.add(B2)).rename("NDWI")

    # NDSI: Normalized Difference Sand Index
    ndsi = B4.subtract(B6).divide(B4.add(B6)).rename("NDSI")

    # MBWI: Modified Bare soil and Water Index
    mbwi = (
        B4.multiply(2)
          .subtract(B1)
          .subtract(B2)
          .subtract(B6)
          .subtract(B7)
          .rename("MBWI")
    )

    return img.addBands([nddi, mbsci, ndvi, ndwi, ndsi, mbwi])


print("Helpers defined.")

Helpers defined.


## 3. Build Image Collection & Select Analysis Date

In [65]:
# ======================================================
# Study period
# ======================================================
start = "2022-01-01"
end   = "2022-12-31"

ic = (
    mod09
    .filterDate(start, end)
    .filterBounds(roi)
    .map(scale_and_mask_MOD09GA)
    .map(add_indices)
)

# ======================================================
# Select a dust event date (April 2022 — known active SDS season)
# ======================================================
date = ee.Date('2022-04-18')

day_col = ic.filterDate(date, date.advance(1, 'day'))
print('Images found for', '2022-04-18', ':', day_col.size().getInfo())

img = ee.Image(day_col.sort('system:time_start').first()).clip(roi)

Images found for 2022-04-18 : 1


## 4. Spectral Indices Maps

Three most important indices for SDS detection:

| Index | Formula | Role in SDS |
|-------|---------|-------------|
| **NDDI** | (SWIR3 − Blue) / (SWIR3 + Blue) | Primary dust aerosol detector — high values = active dust |
| **MBSCI** | (Green + Red) / (SWIR2 + SWIR3) | Identifies bare/sandy surfaces (sediment source areas) |
| **NDVI** | (NIR − Red) / (NIR + Red) | Vegetation cover — low values = exposed dust source |


In [66]:

# ======================================================
# Map 1: NDDI — Normalized Difference Dust Index
# Higher values indicate active dust/sand storm presence
# ======================================================
nddi_map = geemap.Map(basemap="HYBRID")
nddi_map.centerObject(roi, 9)

nddi_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
nddi_map.addLayer(
    img.select('NDDI'),
    {
        'min': -0.3,
        'max':  0.6,
        'palette': ['#2166ac', '#92c5de', '#f7f7f7', '#fddbc7', '#d6604d', '#b2182b']
    },
    'NDDI (Dust Index)'
)

nddi_map.add_colorbar(
    vis_params={'min': -0.3, 'max': 0.6,
                'palette': ['#2166ac', '#92c5de', '#f7f7f7', '#fddbc7', '#d6604d', '#b2182b']},
    label='NDDI',
    orientation='horizontal',
    layer_name='NDDI (Dust Index)'
)

print("NDDI — higher (red) = active dust/sand aerosol")
nddi_map



NDDI — higher (red) = active dust/sand aerosol


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

In [67]:

# ======================================================
# Map 2: MBSCI — Modified Bare Soil Composition Index
# Identifies bare sandy/dusty surfaces (sediment source areas)
# ======================================================
mbsci_map = geemap.Map(basemap="HYBRID")
mbsci_map.centerObject(roi, 9)

mbsci_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
mbsci_map.addLayer(
    img.select('MBSCI'),
    {
        'min':  0.3,
        'max':  1.5,
        'palette': ['#4d9221', '#a1d76a', '#f7f7f7', '#e9a3c9', '#c51b7d']
    },
    'MBSCI (Bare Soil)'
)
mbsci_map.add_colorbar(
    vis_params={'min': 0.3, 'max': 1.5,
                'palette': ['#4d9221', '#a1d76a', '#f7f7f7', '#e9a3c9', '#c51b7d']},
    label='MBSCI',
    orientation='horizontal',
    layer_name='MBSCI (Bare Soil)'
)

print("MBSCI — higher (pink/purple) = more exposed bare/sandy soil")
mbsci_map


MBSCI — higher (pink/purple) = more exposed bare/sandy soil


Map(center=[24.94287090991497, 55.38817725705568], controls=(WidgetControl(options=['position', 'transparent_b…

In [68]:

# ======================================================
# Map 3: NDVI — Normalized Difference Vegetation Index
# Low NDVI = exposed arid surface = higher SDS susceptibility
# ======================================================
ndvi_map = geemap.Map(basemap="HYBRID")
ndvi_map.centerObject(roi, 9)

ndvi_map.addLayer(
    roi,
    {'color': 'grey'},
    'Dubai boundary'
)
ndvi_map.addLayer(
    img.select('NDVI'),
    {
        'min': -0.1,
        'max':  0.5,
        'palette': ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e']
    },
    'NDVI (Vegetation)'
)
ndvi_map.add_colorbar(
    vis_params={'min': -0.1, 'max': 0.5,
                'palette': ['#8c510a', '#d8b365', '#f6e8c3', '#c7eae5', '#5ab4ac', '#01665e']},
    label='NDVI',
    orientation='horizontal',
    layer_name='NDVI (Vegetation)'
)

print("NDVI — brown/low = arid exposed surface (high SDS risk); green/high = vegetated")
ndvi_map


NDVI — brown/low = arid exposed surface (high SDS risk); green/high = vegetated


Map(center=[24.942870909915, 55.3881772570556], controls=(WidgetControl(options=['position', 'transparent_bg']…

## 5. Enhanced Dust Index (EDI)

EDI is computed in [`edi.py`](edi.py) following the methodology of the reference paper.  
Formula: **EDI = NDDi − NDVI**, where NDDi = (SWIR3 − Blue) / (SWIR3 + Blue).  
High positive EDI → active dust/sand aerosol over exposed surface.

In [70]:

# Import EDI module (see edi.py for full methodology comments)
import importlib, sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import edi as edi_module
importlib.reload(edi_module)   # picks up any edits without restarting kernel

from edi import compute_edi, classify_edi

# Apply EDI to the already-clipped single-day image
img_edi        = compute_edi(img)
edi_band       = img_edi.select("EDI")
edi_class_band = classify_edi(edi_band)

print("EDI bands available:", edi_band.bandNames().getInfo())


EDI bands available: ['EDI']


In [71]:

# ======================================================
# Map: Enhanced Dust Index (EDI)
# ======================================================
edi_map = geemap.Map(basemap="HYBRID")
edi_map.centerObject(roi, 9)

edi_map.addLayer(roi, {'color': 'white'}, 'Dubai boundary')

edi_vis = {
    'min': -0.4,
    'max':  0.5,
    'palette': ['#4575b4', '#91bfdb', '#e0f3f8', '#ffffbf', '#fc8d59', '#d73027']
}
edi_map.addLayer(edi_band, edi_vis, 'EDI')

# Single colorbar — no layer_name to avoid geemap auto-duplicating legends
edi_map.add_colorbar(
    vis_params=edi_vis,
    label='EDI  (blue = non-dust / vegetation   →   red = active dust)',
    orientation='horizontal'
)

edi_map


Map(center=[24.94287090991497, 55.38817725705568], controls=(WidgetControl(options=['position', 'transparent_b…